# MOKE Image Analysis with ROI Background Subtraction

This notebook loads DC, Kerr, and RMCD measurement data from a QCoDeS database, visualizes the images, and allows interactive ROI selection for background subtraction.

In [1]:
# Import Required Libraries
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.widgets import PolygonSelector
from skimage.draw import polygon2mask
from tkinter import Tk, filedialog
import qcodes as qc
from qcodes import initialise_or_create_database_at
%matplotlib qt
#%matplotlib auto For Spyder

In [2]:
# Load Data from QCoDeS Database
root = Tk()
root.withdraw()  # Hide the root window

db_file_path = filedialog.askopenfilename(title="Select a QCoDeS database file")
if db_file_path:
    print("Selected file:", db_file_path)
    root.destroy()
else:
    print("No file selected")
    raise RuntimeError("No database file selected.")

qc.initialise_or_create_database_at(db_file_path)


Selected file: /home/main-user/Downloads/data/CrS2/CrS2_Dec_2025.db


In [209]:
nID = int(input("Define nID: "))
print("ID =", nID)
data = qc.load_by_id(nID)

ID = 112


In [210]:
# Extract and Reshape Measurement Data
try:
    dcX = data.get_parameter_data('dcli_dcli_buffer_X')['dcli_dcli_buffer_X']['dcli_dcli_buffer_X']
    f1X = data.get_parameter_data('f1li_f1li_buffer_X')['f1li_f1li_buffer_X']['f1li_f1li_buffer_X']
    f2X = data.get_parameter_data('f2li_f2li_buffer_X')['f2li_f2li_buffer_X']['f2li_f2li_buffer_X']
except:
    pass
try:
    dcX = data.get_parameter_data('dcli_ch1_databuffer')['dcli_ch1_databuffer']['dcli_ch1_databuffer']
    f1X = data.get_parameter_data('f1li_ch1_databuffer')['f1li_ch1_databuffer']['f1li_ch1_databuffer']
    f2X = data.get_parameter_data('f2li_ch1_databuffer')['f2li_ch1_databuffer']['f2li_ch1_databuffer']
except:
    pass
try:
    xsize = data.get_parameter_data('dcli_X')['dcli_X']['scanners_axis1_offset']
    ysize = data.get_parameter_data('dcli_X')['dcli_X']['scanners_axis2_offset']
    if xsize[1]-xsize[0] is not np.inf:
        xshape = int((np.max(xsize)-np.min(xsize))/(xsize[1] - xsize[0]) + 1)
        yshape = int((np.max(ysize)-np.min(ysize))/(ysize[xshape] - ysize[0]) + 1)
    elif ysize[1]-ysize[0] is not np.inf:
        yshape = int((np.max(ysize)-np.min(ysize))/(ysize[1] - ysize[0]) + 1)
        xshape = int((np.max(xsize)-np.min(xsize))/(xsize[xshape] - xsize[0]) + 1)
    shape = (xshape, yshape)
    dcX = np.reshape(data.get_parameter_data('dcli_X')['dcli_X']['dcli_X'],shape)
    f1X = np.reshape(data.get_parameter_data('f1li_X')['f1li_X']['f1li_X'],shape)
    f2X = np.reshape(data.get_parameter_data('f2li_X')['f2li_X']['f2li_X'],shape)
except:
    pass
RMCD = f1X / dcX
Kerr = f2X / dcX

# Plot Raw DC, Kerr, and RMCD Images
redblue = LinearSegmentedColormap.from_list("redblue", ["blue", "white", "red"])
x = np.linspace(0, 32, 5)
y = np.linspace(0, 32, 5)

normalized_path = os.path.normpath(db_file_path)
path = os.path.dirname(normalized_path)
path = os.path.join(path, str(nID))
if not os.path.exists(path):
    os.mkdir(path)

fig, ax = plt.subplots(figsize=(5.75, 4.2))
im_dc = ax.imshow(dcX.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap='gray', aspect='auto')
cbar_dc = plt.colorbar(im_dc, ax=ax)
cbar_dc.ax.tick_params(labelsize=20, colors='black')
cbar_dc.set_label("V", fontsize=20, color='black')
ax.set_xlabel("um", fontsize=20, color='black')
ax.set_ylabel("um", fontsize=20, color='black')
ax.set_title("DC")
ax.tick_params(axis='both', labelsize= 'xx-large', length=10, which='major')
plt.tight_layout()
plt.show()
fig.savefig(path + '/' + 'ID' + str(nID)+ 'DC' +'.png')


fig, axes = plt.subplots(1, 2, figsize=(17, 4.2))

im_kerr = axes[0].imshow(Kerr.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto')
cbar_kerr = plt.colorbar(im_kerr, ax=axes[0])
cbar_kerr.set_label("a.u.")
axes[0].set_xlabel("um")
axes[0].set_ylabel("um")
axes[0].set_title("Kerr")
axes[0].tick_params(labelsize=12)

im_rmcd = axes[1].imshow(RMCD.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto')
cbar_rmcd = plt.colorbar(im_rmcd, ax=axes[1])
cbar_rmcd.set_label("a.u.")
axes[1].set_xlabel("um")
axes[1].set_ylabel("um")
axes[1].set_title("RMCD")
axes[1].tick_params(labelsize=12)

plt.tight_layout()
plt.show()

# Define ROISelector Class for Polygon ROI
class ROISelector:
    def __init__(self, ax, image_shape, x_extent, y_extent):
        self.ax = ax
        self.image_shape = image_shape
        self.x_extent = x_extent
        self.y_extent = y_extent
        self.verts = None
        self.selector = PolygonSelector(
            ax, self.onselect,
            props=dict(color='r', linestyle='-', linewidth=2, alpha=0.5),
            handle_props=dict(marker='o', markersize=5, mec='r', mfc='r', alpha=0.5)
        )
        #self.selector.set_closed(True)
        self.done = False
        self.cid = plt.connect('button_press_event', self.on_double_click)
        print("Draw ROI polygon with mouse. Double-click to close polygon.")
        plt.show()
    def onselect(self, verts):
        self.verts = verts
    def on_double_click(self, event):
        if event.dblclick:
            plt.close()
    def get_mask(self):
        if self.verts is not None:
            arr_verts = []
            # Map display coordinates to array indices for transposed display
            for x_disp, y_disp in self.verts:
                col = int(np.round((x_disp - self.x_extent[0]) / (self.x_extent[1] - self.x_extent[0]) * (self.image_shape[1] - 1)))
                row = int(np.round((y_disp - self.y_extent[0]) / (self.y_extent[1] - self.y_extent[0]) * (self.image_shape[0] - 1)))
                arr_verts.append([row, col])
            mask = polygon2mask(self.image_shape, np.array(arr_verts))
            return mask
        else:
            return None

In [211]:
# Select ROI for Kerr background subtraction (draw polygon, double-click to close)
print("Draw ROI for Kerr background subtraction (draw polygon, double-click to close)")
fig_kerr, ax_kerr = plt.subplots(figsize=(5.75, 4.2))
ax_kerr.imshow(Kerr, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto')
plt.tight_layout()
plt.title("Kerr select ROI on substrate")
roi_kerr = ROISelector(ax_kerr, Kerr.shape, [x[0], x[-1]], [y[0], y[-1]])
# After drawing ROI, run the next cell to perform background subtraction.

Draw ROI for Kerr background subtraction (draw polygon, double-click to close)
Draw ROI polygon with mouse. Double-click to close polygon.


In [212]:
# Perform Kerr background subtraction using selected ROI
mask_kerr = roi_kerr.get_mask()
if mask_kerr is not None:
    substrate_average_k = Kerr[mask_kerr].mean()
    Kerr_bc = Kerr - substrate_average_k
    print(f"Kerr substrate average: {substrate_average_k}")
else:
    print("No ROI selected for Kerr!")
    Kerr_bc = Kerr

Kerr substrate average: -0.4327498072988192


In [7]:
# # Overlay interpolated mask area (ROI) on top of interpolated Kerr plot for visualization (square aspect)
# from skimage.transform import resize
# if mask_kerr is not None and roi_kerr.verts is not None:
#     # Interpolate both mask and Kerr image to square grid
#     square_size = max(mask_kerr.shape)
#     mask_sq = resize(mask_kerr, (square_size, square_size), order=0, preserve_range=True, anti_aliasing=False).astype(bool)
#     Kerr_sq = resize(Kerr, (square_size, square_size), order=1, preserve_range=True, anti_aliasing=True)
#     fig, ax = plt.subplots(figsize=(7, 7))
#     im = ax.imshow(Kerr_sq.T, origin='lower', cmap=redblue, aspect='equal')
#     # Overlay mask as semi-transparent green, with aspect='equal'
#     ax.imshow(np.ma.masked_where(~mask_sq.T, mask_sq.T), origin='lower', cmap='Greens', alpha=0.4, aspect='equal')
#     ax.set_title('Interpolated Kerr Image with ROI Mask Overlay (Square Aspect)')
#     plt.colorbar(im, ax=ax)
#     plt.tight_layout()
#     plt.show()
# else:
#     print('No mask or ROI vertices available. Draw ROI and run previous cells first.')

In [213]:
# Plot Background-Corrected Kerr Image
fig, ax = plt.subplots(figsize=(5.75, 4.2))
im = ax.imshow(Kerr_bc.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto', vmin=-0.03, vmax=0.03)
cbar = plt.colorbar(im, ax=ax)
cbar.ax.tick_params(labelsize=20, colors='black')
cbar.set_label("Kerr (a.u.)", fontsize=20, color='black')
ax.set_xlabel("um", fontsize=20,  color='black')
ax.set_ylabel("um", fontsize=20, color='black')
ax.set_title("Kerr (background corrected)")
ax.tick_params(axis='both', labelsize= 'xx-large', length=10, which='major')
ax.set_xlim(x[0], x[-1])
ax.set_ylim(y[0], y[-1])
plt.tight_layout()
plt.show()
fig.savefig(path + '/' + 'ID' + str(nID)+ 'Kerr' +'.png')

In [214]:
# Average of the flake after subtraction
# Select ROI for Kerr flake subtraction (draw polygon, double-click to close)
print("Draw ROI for Kerr flake subtraction (draw polygon, double-click to close)")
fig_kerr_flake, ax_kerr_flake = plt.subplots(figsize=(5.75, 4.2))
ax_kerr_flake.imshow(Kerr_bc, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto', vmin=-0.06, vmax=0.06)
plt.tight_layout()
plt.title("Kerr select ROI on flake")
roi_kerr_flake = ROISelector(ax_kerr_flake, Kerr_bc.shape, [x[0], x[-1]], [y[0], y[-1]])

Draw ROI for Kerr flake subtraction (draw polygon, double-click to close)
Draw ROI polygon with mouse. Double-click to close polygon.


In [215]:
# Perform Kerr background subtraction using selected ROI
mask_kerr_flake = roi_kerr_flake.get_mask()
if mask_kerr_flake is not None:
    flake_average_kerr = Kerr_bc[mask_kerr_flake].mean()
    print(f"Kerr flake average: {flake_average_kerr}")
else:
    print("No ROI selected for Kerr!")
    Kerr_bc = Kerr

Kerr flake average: -0.009416933261301523


ID T (K) H(T) kerr sig
53 120 +3 0.02516
54 120 0 -0.00077
55 120 -3 -0.02397

67 300 +3 -0.044266
68 300 0 0.006098
70 300 -3 0.043524

76 13 +3 0.15190
77 13 -3 -0.12027
78 13 0 0.002198
83 248 +3 -0.011323
84 248 -3 0.0099535
85 248 0 0.0013968
87 235 +3 -0.01314
88 247 -3 0.010745
89 247 0 0.00213
94 50 +3 -0.0108
95 50 -3 0.009479
92 50 0 0.000354
100 100 +3 -0.01165
101 100 -3 0.010193
98 100 0 -0.000138
104 150 +3 -0.00987
107 150 -3 0.00927
102 150 0 -0.000107
111 200 +3 0.01207
112 200 -3 -0.009417
109 200 0 -0.000104

In [11]:
# # Overlay interpolated mask area (ROI) on top of interpolated Kerr plot for visualization (square aspect)
# from skimage.transform import resize
# if mask_kerr_flake is not None and roi_kerr_flake.verts is not None:
#     # Interpolate both mask and Kerr image to square grid
#     square_size = max(mask_kerr_flake.shape)
#     mask_sq = resize(mask_kerr_flake, (square_size, square_size), order=0, preserve_range=True, anti_aliasing=False).astype(bool)
#     Kerr_bc_sq = resize(Kerr_bc, (square_size, square_size), order=1, preserve_range=True, anti_aliasing=True)
#     fig, ax = plt.subplots(figsize=(7, 7))
#     im = ax.imshow(Kerr_bc_sq.T, origin='lower', cmap=redblue, aspect='equal')
#     # Overlay mask as semi-transparent green, with aspect='equal'
#     ax.imshow(np.ma.masked_where(~mask_sq.T, mask_sq.T), origin='lower', cmap='Greens', alpha=0.4, aspect='equal')
#     ax.set_title('Interpolated Kerr Image with ROI Mask Overlay (Square Aspect)')
#     plt.colorbar(im, ax=ax)
#     plt.tight_layout()
#     plt.show()
# else:
#     print('No mask or ROI vertices available. Draw ROI and run previous cells first.')

In [12]:
# Select ROI for RMCD background subtraction (draw polygon, double-click to close)
print("Draw ROI for RMCD background subtraction (draw polygon, double-click to close)")
fig_RMCD, ax_RMCD = plt.subplots(figsize=(5.75, 4.2))
ax_RMCD.imshow(RMCD, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto')
plt.tight_layout()
plt.title("RMCD select ROI on substrate")
roi_RMCD = ROISelector(ax_RMCD, RMCD.shape, [x[0], x[-1]], [y[0], y[-1]])
# After drawing ROI, run the next cell to perform background subtraction.

Draw ROI for RMCD background subtraction (draw polygon, double-click to close)
Draw ROI polygon with mouse. Double-click to close polygon.


In [13]:
# Perform RMCD background subtraction using selected ROI
mask_RMCD = roi_RMCD.get_mask()
if mask_RMCD is not None:
    substrate_average_RMCD = RMCD[mask_RMCD].mean()
    RMCD_bc = RMCD - substrate_average_RMCD
    print(f"RMCD substrate average: {substrate_average_RMCD}")
else:
    print("No ROI selected for RMCD!")
    RMCD_bc = RMCD

No ROI selected for RMCD!


In [14]:
# # Overlay interpolated mask area (ROI) on top of interpolated Kerr plot for visualization (square aspect)
# from skimage.transform import resize
# if mask_RMCD is not None and roi_RMCD.verts is not None:
#     # Interpolate both mask and RMCD image to square grid
#     square_size = max(mask_RMCD.shape)
#     mask_sq = resize(mask_RMCD, (square_size, square_size), order=0, preserve_range=True, anti_aliasing=False).astype(bool)
#     RMCD_sq = resize(RMCD, (square_size, square_size), order=1, preserve_range=True, anti_aliasing=True)
#     fig, ax = plt.subplots(figsize=(7, 7))
#     im = ax.imshow(RMCD_sq.T, origin='lower', cmap=redblue, aspect='equal')
#     # Overlay mask as semi-transparent green, with aspect='equal'
#     ax.imshow(np.ma.masked_where(~mask_sq.T, mask_sq.T), origin='lower', cmap='Greens', alpha=0.4, aspect='equal')
#     ax.set_title('Interpolated RMCD Image with ROI Mask Overlay (Square Aspect)')
#     plt.colorbar(im, ax=ax)
#     plt.tight_layout()
#     plt.show()
# else:
#     print('No mask or ROI vertices available. Draw ROI and run previous cells first.')

In [15]:
# Plot Background-Corrected RMCD Image
fig, ax = plt.subplots(figsize=(5.75, 4.2))
im = ax.imshow(RMCD_bc.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto', vmin=-0.03, vmax=0.03)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("RMCD (a.u.)", fontsize=20, family = 'Arial', color='black')
ax.set_xlabel("nm", fontsize=20, family = 'Arial', color='black')
ax.set_ylabel("nm", fontsize=20, family = 'Arial', color='black')
ax.set_title("RMCD (background corrected)")
ax.tick_params(axis='both', labelsize= 'xx-large', length=10, which='major')
ax.set_xlim(x[0], x[-1])
ax.set_ylim(y[0], y[-1])
plt.tight_layout()
plt.show()
fig.savefig(path + '/' + 'ID' + str(nID)+ 'RMCD' +'.png')

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.


In [16]:
# Average of the flake after subtraction
# Select ROI for RMCD flake subtraction (draw polygon, double-click to close)
print("Draw ROI for RMCD flake subtraction (draw polygon, double-click to close)")
fig_RMCD_flake, ax_RMCD_flake = plt.subplots(figsize=(5.75, 4.2))
ax_RMCD_flake.imshow(RMCD_bc, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto', vmin=-0.05, vmax=0.05)
plt.tight_layout()
plt.title("RMCD select ROI on flake")
roi_RMCD_flake = ROISelector(ax_RMCD_flake, RMCD_bc.shape, [x[0], x[-1]], [y[0], y[-1]])

Draw ROI for RMCD flake subtraction (draw polygon, double-click to close)
Draw ROI polygon with mouse. Double-click to close polygon.


In [17]:
# Perform Kerr background subtraction using selected ROI
mask_RMCD_flake = roi_RMCD_flake.get_mask()
if mask_RMCD_flake is not None:
    flake_average_RMCD = RMCD_bc[mask_RMCD_flake].mean()
    print(f"RMCD flake average: {flake_average_RMCD}")
else:
    print("No ROI selected for RMCD!")
    RMCD_bc = RMCD

No ROI selected for RMCD!


In [18]:
# # Overlay interpolated mask area (ROI) on top of interpolated RMCD plot for visualization (square aspect)
# from skimage.transform import resize
# if mask_RMCD_flake is not None and roi_RMCD_flake.verts is not None:
#     # Interpolate both mask and RMCD image to square grid
#     square_size = max(mask_RMCD_flake.shape)
#     mask_sq = resize(mask_RMCD_flake, (square_size, square_size), order=0, preserve_range=True, anti_aliasing=False).astype(bool)
#     RMCD_bc_sq = resize(RMCD_bc, (square_size, square_size), order=1, preserve_range=True, anti_aliasing=True)
#     fig, ax = plt.subplots(figsize=(7, 7))
#     im = ax.imshow(RMCD_bc_sq.T, origin='lower', cmap=redblue, aspect='equal', vmin=-0.05, vmax=0.05)
#     # Overlay mask as semi-transparent green, with aspect='equal'
#     ax.imshow(np.ma.masked_where(~mask_sq.T, mask_sq.T), origin='lower', cmap='Greens', alpha=0.4, aspect='equal') 
#     ax.set_title('Interpolated RMCD Image with ROI Mask Overlay (Square Aspect)')
#     plt.colorbar(im, ax=ax)
#     plt.tight_layout()
#     plt.show()
# else:
#     print('No mask or ROI vertices available. Draw ROI and run previous cells first.')

In [19]:
# =========================
# Save results to Excel
# =========================
import pandas as pd
results_file = "results.xlsx"   # change path if you want it somewhere fixed

# Collect results in a dict
results_dict = {
    "nID": [nID],
    "Kerr_flake_avg": [flake_average_kerr if mask_kerr_flake is not None else None],
    "RMCD_flake_avg": [flake_average_RMCD if mask_RMCD_flake is not None else None]
}

df_new = pd.DataFrame(results_dict)

# Append if file exists, otherwise create new
if os.path.exists(results_file):
    df_existing = pd.read_excel(results_file)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df_all = df_new

# Write back to Excel
df_all.to_excel(results_file, index=False)

print(f"✅ Results for ID {nID} saved to {results_file}")

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
# Extract and Reshape Measurement Data
try:
    dcX = data.get_parameter_data('dcli_dcli_buffer_X')['dcli_dcli_buffer_X']['dcli_dcli_buffer_X']
    f1X = data.get_parameter_data('f1li_f1li_buffer_X')['f1li_f1li_buffer_X']['f1li_f1li_buffer_X']
    f2X = data.get_parameter_data('f2li_f2li_buffer_X')['f2li_f2li_buffer_X']['f2li_f2li_buffer_X']
    dcY = data.get_parameter_data('dcli_dcli_buffer_Y')['dcli_dcli_buffer_Y']['dcli_dcli_buffer_Y']
    f1Y = data.get_parameter_data('f1li_f1li_buffer_Y')['f1li_f1li_buffer_Y']['f1li_f1li_buffer_Y']
    f2Y = data.get_parameter_data('f2li_f2li_buffer_Y')['f2li_f2li_buffer_Y']['f2li_f2li_buffer_Y']

except:
    pass
try:
    dcX = data.get_parameter_data('dcli_ch1_databuffer')['dcli_ch1_databuffer']['dcli_ch1_databuffer']
    f1X = data.get_parameter_data('f1li_ch1_databuffer')['f1li_ch1_databuffer']['f1li_ch1_databuffer']
    f2X = data.get_parameter_data('f2li_ch1_databuffer')['f2li_ch1_databuffer']['f2li_ch1_databuffer']
    dcY = data.get_parameter_data('dcli_ch2_databuffer')['dcli_ch2_databuffer']['dcli_ch2_databuffer']
    f1Y = data.get_parameter_data('f1li_ch2_databuffer')['f1li_ch2_databuffer']['f1li_ch2_databuffer']
    f2Y = data.get_parameter_data('f2li_ch2_databuffer')['f2li_ch2_databuffer']['f2li_ch2_databuffer']
except:
    pass
try:
    dcX = data.get_parameter_data('dcli_X')['dcli_X']['dcli_X']
    f1X = data.get_parameter_data('f1li_X')['f1li_X']['f1li_X']
    f2X = data.get_parameter_data('f2li_X')['f2li_X']['f2li_X']
    dcX=dcX.reshape(100, 100)
    f1X=f1X.reshape(100, 100)
    f2X=f2X.reshape(100, 100)
except:
    pass
dc = np.sign(dcX) * np.sqrt(dcX**2 + dcY**2)
f1 = np.sign(f1X) * np.sqrt(f1X**2 + f1Y**2)
f2 = np.sign(f2X) * np.sqrt(f2X**2 + f2Y**2)
RMCD = f1 / dc
Kerr = f2 / dc

# Plot Raw DC, Kerr, and RMCD Images
redblue = LinearSegmentedColormap.from_list("redblue", ["blue", "white", "red"])
x = np.linspace(0, 32, 5)
y = np.linspace(0, 32, 5)

normalized_path = os.path.normpath(db_file_path)
path = os.path.dirname(normalized_path)
path = os.path.join(path, str(nID))
if not os.path.exists(path):
    os.mkdir(path)

fig, ax = plt.subplots(figsize=(5.75, 4.2))
im_dc = ax.imshow(dc.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap='gray', aspect='auto')
cbar_dc = plt.colorbar(im_dc, ax=ax)
cbar_dc.ax.tick_params(labelsize=20, colors='black')
cbar_dc.set_label("mV", fontsize=20, family = 'Arial', color='black')
ax.set_xlabel("nm", fontsize=20, family = 'Arial', color='black')
ax.set_ylabel("nm", fontsize=20, family = 'Arial', color='black')
ax.set_title("DC")
ax.tick_params(axis='both', labelsize= 'xx-large', length=10, which='major')
plt.tight_layout()
plt.show()
fig.savefig(path + '/' + 'ID' + str(nID)+ 'DC' +'.png')


fig, axes = plt.subplots(1, 2, figsize=(17, 4.2))

im_kerr = axes[0].imshow(Kerr.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto')
cbar_kerr = plt.colorbar(im_kerr, ax=axes[0])
cbar_kerr.set_label("a.u.")
axes[0].set_xlabel("nm")
axes[0].set_ylabel("nm")
axes[0].set_title("Kerr")
axes[0].tick_params(labelsize=12)

im_rmcd = axes[1].imshow(RMCD.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap=redblue, aspect='auto')
cbar_rmcd = plt.colorbar(im_rmcd, ax=axes[1])
cbar_rmcd.set_label("a.u.")
axes[1].set_xlabel("nm")
axes[1].set_ylabel("nm")
axes[1].set_title("RMCD")
axes[1].tick_params(labelsize=12)

plt.tight_layout()
plt.show()

# Define ROISelector Class for Polygon ROI
class ROISelector:
    def __init__(self, ax, image_shape, x_extent, y_extent):
        self.ax = ax
        self.image_shape = image_shape
        self.x_extent = x_extent
        self.y_extent = y_extent
        self.verts = None
        self.selector = PolygonSelector(
            ax, self.onselect,
            props=dict(color='r', linestyle='-', linewidth=2, alpha=0.5),
            handle_props=dict(marker='o', markersize=5, mec='r', mfc='r', alpha=0.5)
        )
        #self.selector.set_closed(True)
        self.done = False
        self.cid = plt.connect('button_press_event', self.on_double_click)
        print("Draw ROI polygon with mouse. Double-click to close polygon.")
        plt.show()
    def onselect(self, verts):
        self.verts = verts
    def on_double_click(self, event):
        if event.dblclick:
            plt.close()
    def get_mask(self):
        if self.verts is not None:
            arr_verts = []
            # Map display coordinates to array indices for transposed display
            for x_disp, y_disp in self.verts:
                col = int(np.round((x_disp - self.x_extent[0]) / (self.x_extent[1] - self.x_extent[0]) * (self.image_shape[1] - 1)))
                row = int(np.round((y_disp - self.y_extent[0]) / (self.y_extent[1] - self.y_extent[0]) * (self.image_shape[0] - 1)))
                arr_verts.append([row, col])
            mask = polygon2mask(self.image_shape, np.array(arr_verts))
            return mask
        else:
            return None

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
qt.qpa.wayland: Creating a fake screen in order for Qt not to crash
qt.qpa.wayland: Creating a fake screen in order for Qt not to crash
qt.qpa.wayland: Creating a fake screen in order for Qt not to crash


In [ ]:
# figure for CrS2 scans did in december 2025
plt.figure()
plt.plot([7,6,5,4,3,3,2,1,0.5,-0.5,-1,-2,-3,-4,-5,-6,-7], [-0.0142, -0.0073, 0.0030, 0.0087, 0.0107, 0.0121, 0.0117, 0.0077, 0.0040, -0.0050, -0.0077, -0.0122, - 0.0098, 0.0024, 0.0041, 0.0132, 0.0143], '-o', label='296K')
#plt.plot([-7,-6,-5,-4,-3,-2,-1,-0.5,0,0.5,1,2,3,4,5,6,7],[-0.0097,-0.0120,-0.0123, -0.0101,-0.0063,-0.0014,-1e-5, -0.0003, -0.0008,-0.0011,-0.0004,0.0016,0.0057,0.0082,0.0105,0.0133,0.0126], '-o', label='10K nolens')
plt.plot([-7,-6,-5,-4,-3,-2,-1,-0.5,0,0.5,1,2,3,4,5,6,7],[0.0144,0.0119,0.0058,-0.0001,-0.0075,-0.0141,-0.0099,-0.0055,0.0002,0.0057,0.0109,0.0158,0.0141,0.0073,-0.0002,-0.0075,-0.0140
], '-o', label='30K')
plt.plot([-7,-6,-5,-4,-3,-2,-1,-0.5,0,0.5,1,2,3,4,5],[0.0132,0.0102,0.0045,-0.0026,-0.0083,-0.0110,-0.0087,-0.0056,-0.0016,0.0033,0.0073, 0.0126, 0.0124, 0.0090, 0.0011], '-o', label='15K')
#plt.plot([-7,-6,-5,-4,-3,-2,-1,-0.5,0,0.5,1,2,3,4,5,6,7],[-0.0048,-0.0048,-0.0036,-0.0028,-0.0015,-0.0030,-0.0018,-0.0019,-0.0018,-0.0028,-0.0041,-0.0023,0.0004,0.0020,0.0023,0.0063,0.0082], '-o', label='15K halfbeam')
plt.legend(frameon=False)
plt.hlines(0, -7, 7, linestyle='dotted',  color='grey')
plt.vlines(0, -0.013, 0.013, linestyle='dotted',  color='grey')
plt.xlabel('Applied magnetic field (T)')
plt.ylabel('Kerr signal (a.u.)')
plt.title('Kerr signal integral over flake (bg-corrected)')